# Making chloropleth maps in Altair

Here's a quick example of how to make a chloropleth map in Altair.  In this example, we'll work with a fairly large data set of baby names in France from 1900-2019, broken down by department.

To work with geographical data, we'll use the `geopandas`, which loads `pandas` dataframes, but with support for geographical outlines in the `geojson` format.  You can use these dataframes just as you would a regular `pandas` dataframe, but they will include that extra geographical outline data.

To get started, we'll need to import our libraries.

In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd
alt.data_transformers.enable('json') 

pass

# Reading our names data

Now, let's read in our dataset.  The exported data is in CSV format, but with a `;` separator instead of commas.  The INSEE data collapses rare names or where department-level information has been elided (presumably to protect individuals with uncommon names or who were one of the only ones born with that name in a given year).  We'll strip those out.

In [2]:
data_dir="../data/"

In [3]:
names = pd.read_csv(f"{data_dir}dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2053401,2,CAPUCINE,2003,67,6
726751,1,ILAN,2003,77,5
1668652,1,XAVIER,1966,57,47
3575299,2,SOLINE,2020,52,5
1552999,1,SYLVAIN,1966,74,7


# Loading map data

Next, let's load some map data of regions in France using `geopandas`.  These map data come from the [INSEE] and [IGN] and were processed into the `geojson` format we'll need to work with by [Grégoire David].  Here's the [github] repository.

In this example, we'll work with the simplified departments tiles for the Hexagon, but that repository contains higher-resolution versions, the DOM-TOM, and more.

[Grégoire David]: https://gregoiredavid.fr
[INSEE]: http://www.insee.fr/fr/methodes/nomenclatures/cog/telechargement.asp
[IGN]: https://geoservices.ign.fr/adminexpress
[github]: https://github.com/gregoiredavid/france-geojson/

In [4]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
19,21,Côte-d'Or,"MULTIPOLYGON (((4.1819 47.15051, 4.18711 47.13..."
74,74,Haute-Savoie,"POLYGON ((6.80252 45.77837, 6.75551 45.76635, ..."
91,91,Essonne,"POLYGON ((2.22656 48.7761, 2.23298 48.7662, 2...."
24,26,Drôme,"POLYGON ((4.80049 45.29836, 4.8588 45.30895, 4..."
72,72,Sarthe,"POLYGON ((-0.05453 48.382, -0.04463 48.37976, ..."


Notice how `depts` is a geopandas dataframe.  We'll use it just as a regular `pandas` dataframe, but it includes the geometry info we need to be able to draw those regions when we pass them into Altair.  We just need to make sure that when we work with our data, we keep them in a geopandas dataframe and not a plain dataframe if we want to draw the departments.

In the next cell, notice how we do a right-merge to bring in department data into names.  We do this as a merge on `depts` because we need a geopandas dataframe.  Remember, `depts` is a geopandas dataframe, while `names` is a regular dataframe.  If we did a left merge on `names`, we'd end up with a regular pandas dataframe. After this merge, both `names` and `depts` will be geopandas dataframes.

**Hint:** Be careful when you do your data joins here.  It's easy to accidentally merge the wrong way to accidentally create a _much bigger_ dataset.

In [5]:

just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
3106254,58,Nièvre,"POLYGON ((2.87463 47.52042, 2.8489 47.53754, 2...",2,MARTINE,1964,58,44
350800,89,Yonne,"POLYGON ((2.93631 48.16339, 2.93475 48.17882, ...",1,DAVID,1993,89,14
2150418,41,Loir-et-Cher,"POLYGON ((0.84122 48.10306, 0.87589 48.10944, ...",2,CLAUDINE,1961,41,27
3308221,13,Bouches-du-Rhône,"POLYGON ((4.73906 43.92406, 4.82174 43.91283, ...",2,OLGA,1958,13,5
1386182,30,Gard,"POLYGON ((3.37365 44.17076, 3.43083 44.148, 3....",1,RÉMY,1971,30,6


# Show a name over all years

Now we'll choose a name to show across all years.  To that, we'll group all of the names in a department together (squashing the years together) and use the sum.

In [ ]:
grouped = (
    just_names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False)['nombre']
    .sum()
    .rename(columns={'nombre': 'nombre'})
)
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt')  # Add geometry data back in
grouped['nombre'] = pd.to_numeric(grouped['nombre'], errors='coerce').fillna(0)
grouped.head()

TypeError: agg function failed [how->sum,dtype->geometry]

Now let's pick a name and check out how it's distribution over the last 120 years across Metropolitan France.  In this example, I choose the name “Lucien,” which I rather like for some reason.

In [ ]:
name = "LUCIEN"
subset = grouped[grouped.preusuel == name].copy()

alt.Chart(subset).mark_geoshape(
    stroke="#F2F2F2",
    strokeWidth=0.7
).encode(
    color=alt.Color(
        "nombre:Q",
        scale=alt.Scale(type="sqrt", scheme="teals"),
        legend=alt.Legend(title="Births (1900-2019)", orient="right"),
    ),
    tooltip=[
        alt.Tooltip("nom:N", title="Department"),
        alt.Tooltip("code:N", title="Code"),
        alt.Tooltip("nombre:Q", title="Births", format=","),
    ],
).properties(
    width=820,
    height=620,
    title=f"Distribution of {name} across French departments",
).configure_view(stroke=None)

alt.Chart(...)

## Multi-line chart: gender trends per name over time

This section builds the visualization style shown in your screenshot: one mini line chart per name, with male and female trends plotted on the same axes over time.

In [ ]:
import altair as alt
import pandas as pd

# Load and prepare name data for trend lines
trend = pd.read_csv("../data/dpt2020.csv", sep=";")
trend = trend[(trend["preusuel"] != "_PRENOMS_RARES") & (trend["dpt"] != "XX")].copy()

trend["annais"] = trend["annais"].astype(int)
trend["nombre"] = trend["nombre"].astype(int)
trend["gender"] = trend["sexe"].map({1: "Male", 2: "Female"})


selected_names = ["DOMINIQUE", "CAMILLE", "CLAUDE", "ALIX", "CHARLIE", "EDEN"]
trend = trend[trend["preusuel"].isin(selected_names)]

# Aggregate across all departments -> yearly counts per name and gender
trend_year = (
    trend.groupby(["annais", "preusuel", "gender"], as_index=False)["nombre"]
    .sum()
    .rename(columns={"annais": "year", "preusuel": "name", "nombre": "births"})
)

trend_year.head()

,year,name,gender,births
0,1900,ALIX,Female,47
1,1900,ALIX,Male,6
2,1900,CAMILLE,Female,711
3,1900,CAMILLE,Male,1188
4,1900,CLAUDE,Male,626


In [ ]:
color_scale = alt.Scale(domain=["Male", "Female"], range=["#4C66AF", "#C4544B"])
name_order = ["DOMINIQUE", "CAMILLE", "CLAUDE", "ALIX", "CHARLIE", "EDEN"]

multi_line = (
    alt.Chart(trend_year)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("year:Q", title=None, axis=alt.Axis(format="d", tickCount=4, grid=False)),
        y=alt.Y("births:Q", title="Births / year", axis=alt.Axis(grid=False)),
        color=alt.Color("gender:N", scale=color_scale, legend=alt.Legend(orient="top", title=None)),
        tooltip=[
            alt.Tooltip("name:N", title="Name"),
            alt.Tooltip("gender:N", title="Gender"),
            alt.Tooltip("year:Q", title="Year", format="d"),
            alt.Tooltip("births:Q", title="Births", format=","),
        ],
    )
    .properties(width=180, height=140)
    .facet(
        facet=alt.Facet("name:N", sort=name_order, header=alt.Header(title=None, labelFontSize=18, labelPadding=8)),
        columns=3,
    )
    .resolve_scale(y="shared")
    .properties(title="Multi-line chart: gender trends per name over time")
)

multi_line

alt.FacetChart(...)